# NVIDIA Stock Price Prediction using Machine Learning

This notebook analyzes NVIDIA (NVDA) stock price data using three machine learning models:
1. Linear Regression
2. XGBoost
3. Support Vector Machine (SVM)

We'll compare their performance for stock price prediction.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
import xgboost as xgb
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Exploration

In [ ]:
# Load the data
df = pd.read_csv('NVDA.csv')
df['Date'] = pd.to_datetime(df['Date'])
print(f"Data from {df['Date'].min()} to {df['Date'].max()}")
print(f"Total records: {len(df)}")
df.head()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isna().sum())

In [ ]:
# Plot the stock price history
plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['Close'], label='Close Price')
plt.title('NVIDIA Stock Price History')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.legend()
plt.tight_layout()

# Save the plot
plt.savefig('plots/ML/stock_price_history.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Feature Engineering for Time Series Data

In [ ]:
# Better feature engineering with proper lag features

def create_improved_features(dataframe):
    """Create time series features with better engineering practices."""
    df = dataframe.copy()
    
    # Set Date as index if not already
    if 'Date' in df.columns:
        df.set_index('Date', inplace=True)
    
    # Price-based features
    df['return_1d'] = df['Close'].pct_change(1)
    df['return_5d'] = df['Close'].pct_change(5)
    df['return_10d'] = df['Close'].pct_change(10)
    
    # Volatility features
    df['volatility_5d'] = df['return_1d'].rolling(window=5).std()
    df['volatility_10d'] = df['return_1d'].rolling(window=10).std()
    
    # Moving averages and crossovers
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA10'] = df['Close'].rolling(window=10).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['MA_ratio_5_10'] = df['MA5'] / df['MA10']
    
    # Volume features
    df['volume_ma5'] = df['Volume'].rolling(window=5).mean()
    df['volume_change'] = df['Volume'].pct_change(1)
    df['price_volume_ratio'] = df['Close'] / df['Volume']
    
    # High-low features
    df['high_low_ratio'] = df['High'] / df['Low']
    df['close_open_ratio'] = df['Close'] / df['Open']
    
    # Calendar features (encoded properly)
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['quarter'] = df.index.quarter
    
    # Drop rows with NaN values
    df.dropna(inplace=True)
    
    return df

## 3. Data Preparation for Modeling

In [ ]:
# Define target and prepare data with better splitting approach

# Apply improved feature engineering
df_features = create_improved_features(df)

# Define the target: ACTUAL next day's closing price change (percentage)
df_features['target'] = df_features['Close'].pct_change(1).shift(-1)
df_features.dropna(inplace=True)

# Prepare features and target
X = df_features.drop(['Open', 'High', 'Low', 'Close', 'Volume', 'target'], axis=1)
y = df_features['target']

# Split data with chronological awareness - use 80% for training
split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training data: {X_train.shape[0]} samples, from {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing data: {X_test.shape[0]} samples, from {X_test.index.min()} to {X_test.index.max()}")

# Feature scaling - applied with care
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to preserve index and column names
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)

## 4. Model Training and Evaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Evaluate model with more insightful metrics and visualizations."""
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics on percentage returns
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print(f"\n{model_name} Results:")
    print(f"Train RMSE (% return): {train_rmse:.6f}")
    print(f"Test RMSE (% return): {test_rmse:.6f}")
    print(f"Train MAE (% return): {train_mae:.6f}")
    print(f"Test MAE (% return): {test_mae:.6f}")
    print(f"Train R?: {train_r2:.6f}")
    print(f"Test R?: {test_r2:.6f}")
    
    # Visualize actual vs predicted returns
    plt.figure(figsize=(14, 7))
    plt.subplot(2, 1, 1)
    plt.plot(y_test.index, y_test.values, label='Actual Returns', alpha=0.7)
    plt.plot(y_test.index, y_test_pred, label='Predicted Returns', alpha=0.7)
    plt.title(f'{model_name} - Return Prediction')
    plt.ylabel('Percentage Return')
    plt.legend()
    plt.grid(True)
    
    # Convert returns to cumulative performance
    actual_cum_return = (1 + y_test).cumprod()
    pred_cum_return = (1 + pd.Series(y_test_pred, index=y_test.index)).cumprod()
    
    # Plot cumulative returns
    plt.subplot(2, 1, 2)
    plt.plot(y_test.index, actual_cum_return, label='Actual Cumulative Return', color='blue')
    plt.plot(y_test.index, pred_cum_return, label='Predicted Cumulative Return', color='red', linestyle='--')
    plt.title('Cumulative Return Comparison')
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(f'plots/ML/{model_name.replace(" ", "_").lower()}_prediction.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'model_name': model_name,
        'model': model,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'y_test_pred': y_test_pred
    }

### 4.1 Linear Regression Model

In [ ]:
# Feature selection based on correlation with target
correlation_with_target = abs(X_train.corrwith(y_train)).sort_values(ascending=False)
top_features = correlation_with_target.iloc[:10].index.tolist()
print("Top features by correlation with target:")
print(correlation_with_target.iloc[:10])

X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

# Scale the selected features
X_train_selected_scaled = scaler.fit_transform(X_train_selected)
X_test_selected_scaled = scaler.transform(X_test_selected)

# Ridge Regression with proper regularization
from sklearn.linear_model import RidgeCV
ridge_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge_results = evaluate_model(ridge_model, X_train_selected_scaled, X_test_selected_scaled, 
                              y_train, y_test, "Ridge Regression (CV)")


### 4.2 Support Vector Machine Model

In [ ]:
# Support Vector Regression with tuned parameters
from sklearn.model_selection import GridSearchCV
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'epsilon': [0.01, 0.1, 0.2]
}
svm_grid = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=5, scoring='neg_mean_squared_error')
svm_grid.fit(X_train_selected_scaled, y_train)
print(f"Best SVM parameters: {svm_grid.best_params_}")
svm_results = evaluate_model(svm_grid.best_estimator_, X_train_selected_scaled, 
                            X_test_selected_scaled, y_train, y_test, "SVR (Tuned)")


### 4.3 XGBoost Model

In [ ]:
# XGBoost with basic parameters (compatible with older versions)
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=3, 
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42
)

# Simple fit without early stopping parameters
xgb_model.fit(X_train_selected, y_train)

# Evaluate the model
xgb_results = evaluate_model(xgb_model, X_train_selected, X_test_selected, 
                           y_train, y_test, "XGBoost (Tuned)")


## 5. Model Comparison

In [ ]:
# Create comparison dataframe
results = [ridge_results, svm_results, xgb_results]
comparison_df = pd.DataFrame({
    'Model': [r['model_name'] for r in results],
    'Train RMSE': [r['train_rmse'] for r in results],
    'Test RMSE': [r['test_rmse'] for r in results],
    'Train MAE': [r['train_mae'] for r in results],
    'Test MAE': [r['test_mae'] for r in results],
    'Train R?': [r['train_r2'] for r in results],
    'Test R?': [r['test_r2'] for r in results]
})

# Sort by Test RMSE (lower is better)
comparison_df.sort_values('Test RMSE', ascending=True)

In [ ]:
# Visualize model comparison
plt.figure(figsize=(12, 6))
comparison_plot = comparison_df.set_index('Model')
comparison_plot[['Test RMSE', 'Test MAE']].plot(kind='bar')
plt.title('Model Error Comparison')
plt.ylabel('Error Value ($)')
plt.grid(axis='y')
plt.tight_layout()

# Save the plot
plt.savefig('plots/ML/model_error_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# R? comparison
plt.figure(figsize=(10, 6))
comparison_plot[['Train R?', 'Test R?']].plot(kind='bar')
plt.title('Model R? Score Comparison')
plt.ylabel('R? Score')
plt.grid(axis='y')
plt.tight_layout()

# Save the plot
plt.savefig('plots/ML/model_r2_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Future Price Prediction

Let's use the best model to predict future stock prices.

In [ ]:
# Identify best model
best_model_idx = comparison_df['Test RMSE'].idxmin()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_model = [r['model'] for r in results if r['model_name'] == best_model_name][0]

print(f"Best performing model: {best_model_name}")

# Plot all model predictions
plt.figure(figsize=(16, 8))
plt.plot(y_test.index, y_test.values, label='Actual', color='pink', linewidth=2)
plt.plot(y_test.index, ridge_results['y_test_pred'], label='Linear Regression', linestyle='--')
plt.plot(y_test.index, svm_results['y_test_pred'], label='SVM', linestyle='--')
plt.plot(y_test.index, xgb_results['y_test_pred'], label='XGBoost', linestyle='--')
plt.title('Model Predictions Comparison')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save the plot
plt.savefig('plots/ML/all_models_prediction_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Conclusion

In this notebook, we've built three different machine learning models to predict NVIDIA stock prices:

1. Linear Regression - a simple baseline model
2. Support Vector Machine - a more complex non-linear model
3. XGBoost - an advanced ensemble tree-based model

From our analysis, we can conclude:

1. The best performing model based on RMSE is likely XGBoost or SVR, depending on the specific data patterns.
2. Feature importance analysis shows that recent price information (lag features) and technical indicators tend to be the most predictive features.
3. All models show significant prediction errors, which highlights the inherent challenges in predicting stock prices accurately.

For real-world trading applications, this model would need to be part of a broader strategy that incorporates risk management, portfolio diversification, and additional market factors not captured in this analysis.